# Unidad 3. Evidencia de Aprendizaje EA3
## Taller: procesamiento de datos en una infraestructura cloud

**Estudiante:** Pablo Emilio G?mez G?mez  
**Programa:** Ingenier?a de Software y Datos  
**Asignatura:** Big Data  
**Fecha:** Bogot?, D. C., 24 de mayo de 2026  
**Notebook:** Gomez_Pablo_Actividad_2.ipynb  
**Repositorio:** Actividad_2  

Este notebook desarrolla la Evidencia de Aprendizaje EA3 mediante Databricks Community Edition, Apache Spark, PySpark y Spark SQL. La pr?ctica utiliza el dataset Titanic de Kaggle para dise?ar un esquema de almacenamiento, cargar datos en DBFS, crear una tabla y validar el procesamiento desde Spark y SQL.

## 1. Descripci?n del problema

El trabajo con Big Data exige algo m?s que almacenar archivos. Tambi?n requiere una infraestructura que permita cargar datos, definir esquemas, consultar informaci?n y validar resultados de forma reproducible. En esta pr?ctica se aborda ese proceso desde un entorno cloud acad?mico, usando Databricks Community Edition como plataforma de ejecuci?n.

El problema consiste en tomar un conjunto de datos externo, estructurarlo con tipos definidos, almacenarlo como tabla y comprobar su funcionamiento mediante consultas equivalentes en PySpark y Spark SQL. Con ello se evidencia el ciclo b?sico de procesamiento de datos en una infraestructura cloud.

## 2. Objetivos

### Objetivo general

Procesar un conjunto de datos en Databricks Community Edition mediante Apache Spark, dise?ando el esquema de almacenamiento, cargando datos desde Kaggle y validando la informaci?n con PySpark y SQL.

### Objetivos espec?ficos

1. Dise?ar un esquema de datos para representar correctamente las variables del dataset Titanic.
2. Evidenciar la configuraci?n del entorno Databricks CE.
3. Cargar el archivo CSV en DBFS o FileStore.
4. Crear una tabla administrada en Spark.
5. Ejecutar validaciones con PySpark y Spark SQL.
6. Comparar ventajas y desventajas de SQL frente a Spark.

## 3. Dataset seleccionado

Se utiliza el dataset Titanic de Kaggle porque es gratuito, peque?o, manejable y suficiente para validar operaciones b?sicas de Big Data. Contiene variables num?ricas y categ?ricas, datos nulos y campos ?tiles para consultas de conteo, descripci?n, agrupaci?n y filtrado.

La ruta esperada del archivo en Databricks ser?:

/FileStore/tables/Titanic_Dataset.csv

Si Databricks guarda el archivo con otro nombre, se debe modificar la variable ruta_csv en la celda de configuraci?n.

## 4. Dise?o del esquema de datos

La tabla propuesta ser?:

ea3_bigdata.ea3_titanic_pasajeros

La llave l?gica del dataset ser? PassengerId, porque identifica de forma ?nica a cada pasajero. Aunque Spark no impone llave primaria como un motor relacional tradicional, esta columna se documenta como identificador principal.

### 4.1 Diccionario de datos

| Campo | Tipo Spark | Nulabilidad | Descripci?n |
|---|---|---|---|
| PassengerId | IntegerType | No | Identificador ?nico del pasajero. |
| Survived | IntegerType | No | Variable de supervivencia: 0 no sobrevivi?, 1 sobrevivi?. |
| Pclass | IntegerType | No | Clase del tiquete: 1, 2 o 3. |
| Name | StringType | No | Nombre del pasajero. |
| Sex | StringType | No | Sexo registrado del pasajero. |
| Age | DoubleType | S? | Edad del pasajero. Puede contener valores nulos. |
| SibSp | IntegerType | No | N?mero de hermanos o c?nyuges a bordo. |
| Parch | IntegerType | No | N?mero de padres o hijos a bordo. |
| Ticket | StringType | S? | N?mero del tiquete. |
| Fare | DoubleType | S? | Tarifa pagada por el pasajero. |
| Cabin | StringType | S? | Cabina asignada. Puede contener muchos valores nulos. |
| Embarked | StringType | S? | Puerto de embarque: C, Q o S. |

### 4.2 Diagrama l?gico simple

Tabla: ea3_titanic_pasajeros

PassengerId funciona como identificador l?gico principal.  
Las dem?s columnas describen caracter?sticas del pasajero, informaci?n del viaje, condiciones familiares, tarifa, cabina y puerto de embarque.

Estructura general:

ea3_titanic_pasajeros  
- PassengerId  
- Survived  
- Pclass  
- Name  
- Sex  
- Age  
- SibSp  
- Parch  
- Ticket  
- Fare  
- Cabin  
- Embarked

In [ ]:
# ============================================================
# CELDA 1 - IMPORTAR LIBRER?AS Y DEFINIR ESQUEMA
# ============================================================
# C?mo se lee:
#   Importamos los tipos de datos de Spark y declaramos
#   el esquema que tendr? el archivo Titanic.
#
# Para qu? sirve:
#   Evita que Spark adivine los tipos de datos.
# ============================================================

from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, DoubleType, StringType
)

schema_titanic = StructType([
    StructField("PassengerId", IntegerType(), False),
    StructField("Survived", IntegerType(), False),
    StructField("Pclass", IntegerType(), False),
    StructField("Name", StringType(), False),
    StructField("Sex", StringType(), False),
    StructField("Age", DoubleType(), True),
    StructField("SibSp", IntegerType(), False),
    StructField("Parch", IntegerType(), False),
    StructField("Ticket", StringType(), True),
    StructField("Fare", DoubleType(), True),
    StructField("Cabin", StringType(), True),
    StructField("Embarked", StringType(), True)
])

print("Esquema definido correctamente.")

## 5. Configuraci?n y evidencia del entorno Databricks CE

Las siguientes celdas dejan evidencia de la configuraci?n del entorno. Se imprimen versiones de Python, Spark, nombre de aplicaci?n, master configurado, SparkContext y rutas de almacenamiento disponibles.

In [ ]:
# ============================================================
# CELDA 2 - VERSIONES DEL ENTORNO
# ============================================================
# C?mo se lee:
#   Consultamos versiones principales del ambiente.
#
# Para qu? sirve:
#   Esta salida sirve como evidencia de configuraci?n.
# ============================================================

import sys
import platform

print("Versi?n de Python:")
print(sys.version)

print("Sistema base:")
print(platform.platform())

print("Versi?n de Apache Spark:")
print(spark.version)

print("Nombre de la aplicaci?n Spark:")
print(spark.sparkContext.appName)

print("Master configurado:")
print(spark.sparkContext.master)

In [ ]:
# ============================================================
# CELDA 3 - CONFIGURACI?N DEL SPARKCONTEXT
# ============================================================
# C?mo se lee:
#   Pedimos a Spark que muestre su configuraci?n activa.
#
# Para qu? sirve:
#   Evidencia par?metros del entorno de ejecuci?n.
# ============================================================

for clave, valor in spark.sparkContext.getConf().getAll():
    print(clave, "=", valor)

In [ ]:
# ============================================================
# CELDA 4 - VALIDAR ALMACENAMIENTO DBFS / FILESTORE
# ============================================================
# C?mo se lee:
#   Listamos rutas disponibles en FileStore.
#
# Para qu? sirve:
#   Comprueba el espacio donde se cargar? el CSV.
# ============================================================

print("Contenido de /FileStore:")
try:
    display(dbutils.fs.ls("/FileStore"))
except Exception as error:
    print("No fue posible listar /FileStore.")
    print(error)

print("Contenido de /FileStore/tables:")
try:
    display(dbutils.fs.ls("/FileStore/tables"))
except Exception as error:
    print("No fue posible listar /FileStore/tables. Si a?n no se subi? el CSV, es normal.")
    print(error)

## 6. Ingesta del archivo CSV

Para facilitar el desarrollo acad?mico, se usar? la carga manual del archivo CSV en Databricks. El archivo debe quedar disponible en FileStore o DBFS. La ruta esperada ser?:

/FileStore/tables/Titanic_Dataset.csv

Si el nombre real queda diferente, solo se cambia la variable ruta_csv.

In [ ]:
# ============================================================
# CELDA 5 - DEFINIR RUTA, BASE DE DATOS Y TABLA
# ============================================================
# C?mo se lee:
#   Guardamos en variables la ruta del CSV y los nombres
#   de la base y la tabla.
#
# Para qu? sirve:
#   Centraliza los nombres para que el notebook sea f?cil de ajustar.
# ============================================================

ruta_csv = "/FileStore/tables/Titanic_Dataset.csv"

base_datos = "ea3_bigdata"
tabla_titanic = "ea3_titanic_pasajeros"
tabla_completa = base_datos + "." + tabla_titanic

print("Ruta CSV:", ruta_csv)
print("Base de datos:", base_datos)
print("Tabla:", tabla_completa)

In [ ]:
# ============================================================
# CELDA 6 - LEER CSV CON ESQUEMA DEFINIDO
# ============================================================
# C?mo se lee:
#   Spark lee el archivo CSV usando el esquema dise?ado.
#
# Para qu? sirve:
#   Convierte el archivo externo en un DataFrame de Spark.
# ============================================================

df_titanic = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .schema(schema_titanic)
    .load(ruta_csv)
)

df_titanic.cache()

print("Archivo le?do correctamente.")
print("Cantidad de registros:", df_titanic.count())

display(df_titanic.limit(10))

In [ ]:
# ============================================================
# CELDA 7 - VALIDAR ESQUEMA Y MUESTRA
# ============================================================
# C?mo se lee:
#   Imprimimos estructura y primeras filas.
#
# Para qu? sirve:
#   Confirma que los tipos de datos se aplicaron correctamente.
# ============================================================

df_titanic.printSchema()
df_titanic.show(5, truncate=False)

In [ ]:
# ============================================================
# CELDA 8 - CREAR BASE DE DATOS Y TABLA
# ============================================================
# C?mo se lee:
#   Creamos una base acad?mica y guardamos el DataFrame
#   como tabla administrada.
#
# Para qu? sirve:
#   Permite consultar el dataset mediante Spark SQL.
# ============================================================

spark.sql("CREATE DATABASE IF NOT EXISTS " + base_datos)
spark.sql("DROP TABLE IF EXISTS " + tabla_completa)

df_titanic.write.mode("overwrite").saveAsTable(tabla_completa)

print("Tabla creada correctamente:", tabla_completa)

In [ ]:
# ============================================================
# CELDA 9 - CONFIRMAR TABLA CREADA
# ============================================================
# C?mo se lee:
#   Mostramos las tablas de la base ea3_bigdata.
#
# Para qu? sirve:
#   Deja evidencia de que la tabla qued? registrada.
# ============================================================

spark.sql("SHOW TABLES IN " + base_datos).show(truncate=False)

## 7. Validaciones con PySpark

En esta secci?n se realizan validaciones con la API de PySpark: esquema, conteo, descripci?n de datos, nulos, agrupaciones y filtros.

In [ ]:
# ============================================================
# CELDA 10 - ESTAD?STICAS DESCRIPTIVAS
# ============================================================
# C?mo se lee:
#   Spark calcula estad?sticas b?sicas del dataset.
#
# Para qu? sirve:
#   Permite revisar conteos, promedios, m?nimos y m?ximos.
# ============================================================

df_titanic.describe().show()

In [ ]:
# ============================================================
# CELDA 11 - CONTEO DE NULOS POR COLUMNA
# ============================================================
# C?mo se lee:
#   Para cada columna contamos los valores nulos.
#
# Para qu? sirve:
#   Permite evaluar calidad b?sica de datos.
# ============================================================

from pyspark.sql.functions import col, sum as spark_sum, when

conteo_nulos = df_titanic.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_titanic.columns
])

display(conteo_nulos)

In [ ]:
# ============================================================
# CELDA 12 - GROUP BY EN PYSPARK
# ============================================================
# C?mo se lee:
#   Agrupamos pasajeros seg?n supervivencia.
#
# Para qu? sirve:
#   Permite conocer cu?ntos sobrevivieron y cu?ntos no.
# ============================================================

resultado_supervivencia_spark = (
    df_titanic
    .groupBy("Survived")
    .count()
    .orderBy("Survived")
)

display(resultado_supervivencia_spark)

In [ ]:
# ============================================================
# CELDA 13 - GROUP BY POR CLASE Y SEXO
# ============================================================
# C?mo se lee:
#   Agrupamos por clase del tiquete y sexo.
#
# Para qu? sirve:
#   Permite observar distribuci?n por variables categ?ricas.
# ============================================================

resultado_clase_sexo_spark = (
    df_titanic
    .groupBy("Pclass", "Sex")
    .count()
    .orderBy("Pclass", "Sex")
)

display(resultado_clase_sexo_spark)

In [ ]:
# ============================================================
# CELDA 14 - FILTRO EN PYSPARK
# ============================================================
# C?mo se lee:
#   Filtramos adultos sobrevivientes.
#
# Para qu? sirve:
#   Evidencia una consulta condicional con PySpark.
# ============================================================

adultos_sobrevivientes = (
    df_titanic
    .filter((col("Age") >= 18) & (col("Survived") == 1))
    .select("PassengerId", "Name", "Sex", "Age", "Pclass", "Fare")
    .orderBy("Age")
)

display(adultos_sobrevivientes.limit(20))

## 8. Validaciones con Spark SQL

En esta secci?n se realizan consultas equivalentes con SQL para validar metadatos, descripci?n, conteos, muestras, agrupaciones y filtros.

In [ ]:
# ============================================================
# CELDA 15 - DESCRIBE TABLE
# ============================================================
# C?mo se lee:
#   Ejecutamos DESCRIBE TABLE sobre la tabla creada.
#
# Para qu? sirve:
#   Muestra metadatos de la tabla.
# ============================================================

display(spark.sql("DESCRIBE TABLE " + tabla_completa))

In [ ]:
# ============================================================
# CELDA 16 - SHOW CREATE TABLE
# ============================================================
# C?mo se lee:
#   Spark muestra c?mo qued? creada la tabla.
#
# Para qu? sirve:
#   Deja evidencia t?cnica del registro de la tabla.
# ============================================================

display(spark.sql("SHOW CREATE TABLE " + tabla_completa))

In [ ]:
# ============================================================
# CELDA 17 - SELECT Y COUNT EN SQL
# ============================================================
# C?mo se lee:
#   Consultamos conteo total y una muestra.
#
# Para qu? sirve:
#   Valida que la tabla puede consultarse con SQL.
# ============================================================

display(spark.sql("SELECT COUNT(*) AS total_registros FROM " + tabla_completa))

display(spark.sql(
    "SELECT PassengerId, Name, Sex, Age, Pclass, Survived "
    "FROM " + tabla_completa + " LIMIT 10"
))

In [ ]:
# ============================================================
# CELDA 18 - GROUP BY EN SQL
# ============================================================
# C?mo se lee:
#   Agrupamos por supervivencia usando SQL.
#
# Para qu? sirve:
#   Compara resultados con PySpark.
# ============================================================

display(spark.sql(
    "SELECT Survived, COUNT(*) AS total_pasajeros "
    "FROM " + tabla_completa + " "
    "GROUP BY Survived "
    "ORDER BY Survived"
))

In [ ]:
# ============================================================
# CELDA 19 - GROUP BY CLASE Y SEXO EN SQL
# ============================================================
# C?mo se lee:
#   Agrupamos por clase y sexo.
#
# Para qu? sirve:
#   Valida agregaciones categ?ricas con SQL.
# ============================================================

display(spark.sql(
    "SELECT Pclass, Sex, COUNT(*) AS total_pasajeros, "
    "ROUND(AVG(Fare), 2) AS tarifa_promedio, "
    "ROUND(AVG(Age), 2) AS edad_promedio "
    "FROM " + tabla_completa + " "
    "GROUP BY Pclass, Sex "
    "ORDER BY Pclass, Sex"
))

In [ ]:
# ============================================================
# CELDA 20 - FILTRO EN SQL
# ============================================================
# C?mo se lee:
#   Filtramos adultos sobrevivientes usando SQL.
#
# Para qu? sirve:
#   Muestra una consulta condicional equivalente a PySpark.
# ============================================================

display(spark.sql(
    "SELECT PassengerId, Name, Sex, Age, Pclass, Fare "
    "FROM " + tabla_completa + " "
    "WHERE Age >= 18 AND Survived = 1 "
    "ORDER BY Age "
    "LIMIT 20"
))

## 9. Interpretaci?n de resultados

Las validaciones permiten comprobar que el archivo fue le?do correctamente, que el esquema se aplic? a las columnas y que la tabla qued? disponible para consultas. El conteo confirma la carga de registros; la revisi?n de nulos permite identificar campos incompletos; y las agrupaciones por supervivencia, clase y sexo muestran que los datos pueden analizarse desde Spark y desde SQL.

La comparaci?n pr?ctica evidencia que SQL es directo para consultas declarativas, mientras que PySpark resulta m?s flexible para construir transformaciones paso a paso dentro de un flujo de procesamiento.

## 10. Comparaci?n entre SQL y Spark

| Criterio | SQL | Spark / PySpark |
|---|---|---|
| Facilidad | Es m?s sencillo para consultas conocidas y reportes. | Requiere comprender la API de DataFrames. |
| Uso principal | Consultas, filtros, agregaciones y an?lisis declarativo. | Limpieza, transformaci?n, pipelines y procesamiento distribuido. |
| Flexibilidad | Puede ser limitado para l?gica personalizada. | Permite combinar programaci?n con procesamiento de datos. |
| Curva de aprendizaje | M?s amigable para quien conoce bases de datos. | M?s exigente, pero m?s potente para flujos complejos. |
| Experiencia en la pr?ctica | Fue ?til para validar y resumir informaci?n r?pidamente. | Fue ?til para leer el archivo, aplicar esquema y crear la tabla. |

En esta pr?ctica, SQL y PySpark se complementan. SQL facilit? las validaciones r?pidas y PySpark permiti? controlar mejor el proceso t?cnico de lectura, transformaci?n y persistencia.

## 11. Conclusiones

La actividad permiti? desarrollar un flujo b?sico de procesamiento de datos en una infraestructura cloud acad?mica. A partir de un dataset externo fue posible dise?ar un esquema, cargar el archivo en Databricks, crear una tabla y validar la informaci?n mediante Spark y SQL.

Databricks Community Edition permiti? integrar documentaci?n, c?digo y resultados visibles en un mismo notebook. Esta caracter?stica es relevante para Big Data porque facilita la trazabilidad del proceso y permite que otros usuarios revisen c?mo se configur? y ejecut? la pr?ctica.

Finalmente, el ejercicio mostr? que Spark y SQL no deben entenderse como herramientas opuestas. En un entorno de an?lisis de datos, SQL resulta pr?ctico para consultar e interpretar, mientras que PySpark ofrece mayor control para estructurar procesos de transformaci?n y procesamiento.

## 12. Referencias

Apache Software Foundation. (s. f.). Apache Spark documentation. https://spark.apache.org/docs/latest/

Databricks. (s. f.). Databricks documentation. https://docs.databricks.com/

Kaggle. (s. f.). Titanic Dataset. https://www.kaggle.com/datasets/yasserh/titanic-dataset

Project Jupyter. (s. f.). Jupyter Notebook documentation. https://docs.jupyter.org/